Answer the problem in the context of topics covered in SDS. You are **not allowed** to seek or receive help from anyone to solve the problem. To keep the problem at reasonable difficulty, you are also **not allowed** to use LLMs such as ChatGPT, Gemini, Claude and Llama. You can **only use built-in libraries** of Apache Spark. For example, you cannot use Spark packages.

# Problem 1 [25 pts]

In this problem, you will design and implement a function that will return the most similar users of a user based on their commonly listened songs.

## Problem 1a [15 pts]

Critique the output of an LLM below for the given prompt. Give three aspects where it was right. Give three aspects where it was wrong or limitations to its solution. Provide justification for each of your answer.

**Prompt:**
    
Discuss the step-by-step process of how you will transform a set of text into a locally-sensitive hash. When performing approximate nearest neighbor search, a cosine similarity of at least 60% should result in an 80% chance of collision and at most 5% chance of collision if their cosine similarity is at most 40%. Be as detailed as possible. Explain the reasoning for each step. Make sure all parameter values are explicitly stated (not just the equation) and justified. 

You should include and specify the following but you may add more information as necessary:

* Locality-sensitive function that you will use
* Relevant hyperparameters
* Signature size
* Number of bands

**LLM output:**

To transform a set of text into a locally-sensitive hash that satisfies your specific collision probabilities based on cosine similarity, we must use a specific Locality-Sensitive Hashing (LSH) family known as **Random Projection** (often referred to as **SimHash** in text processing).

Because your constraints require a sharp distinction between a $60\%$ similarity ($80\%$ chance of collision) and a $40\%$ similarity ($5\%$ chance of collision), the resulting hyperparameters will force a very steep "S-curve."

Here is the detailed, step-by-step process, including the mathematical justification for the required hyperparameters.

---

### Step 1: Text Preprocessing and Vectorization

Before hashing, text must be converted into a numerical vector space where cosine similarity is meaningful.

1. **Tokenization & Processing:** Remove stop words, stem/lemmatize the text, and convert it into tokens (e.g., words or n-grams).
2. **Vectorization:** Convert the tokens into high-dimensional vectors. For cosine similarity, **TF-IDF (Term Frequency-Inverse Document Frequency)** is the standard approach, though dense embeddings (like Word2Vec or BERT) can also be used.
*Reasoning:* Cosine similarity measures the angle between two vectors. We need a vector representation where semantically or lexically similar texts point in roughly the same direction in the vector space.

### Step 2: Choosing the LSH Function (SimHash / Random Projection)

The LSH function family that approximates Cosine Similarity is **Random Projection**.

* **The Function:** We generate a set of random hyperplanes (vectors) from a standard normal distribution. For a given text vector $\vec{v}$ and a random hyperplane vector $\vec{h}$, the hash function $f(\vec{v})$ is defined as:

$$f(\vec{v}) = \begin{cases} 1 & \text{if } \vec{v} \cdot \vec{h} \ge 0 \\ 0 & \text{if } \vec{v} \cdot \vec{h} < 0 \end{cases}$$


* **Reasoning:** The probability that two vectors $\vec{x}$ and $\vec{y}$ fall on the same side of a random hyperplane (thus yielding the same hash bit) is directly proportional to the angle $\theta$ between them.

The probability of a single hash collision $p$ for cosine similarity $S_c$ is:


$$p = 1 - \frac{\theta}{\pi} = 1 - \frac{\arccos(S_c)}{\pi}$$

### Step 3: Determining Base Collision Probabilities

We must calculate the single-bit collision probability $p$ for your two thresholds: $S_c = 0.60$ and $S_c = 0.40$.

* **For $S_c = 0.60$ (Target: $\ge 80\%$ final collision rate):**

$$\theta_1 = \arccos(0.60) \approx 0.9273 \text{ radians}$$


$$p_1 = 1 - \frac{0.9273}{\pi} \approx 0.7048$$


* **For $S_c = 0.40$ (Target: $\le 5\%$ final collision rate):**

$$\theta_2 = \arccos(0.40) \approx 1.1593 \text{ radians}$$


$$p_2 = 1 - \frac{1.1593}{\pi} \approx 0.6310$$



### Step 4: Applying the Banding Technique

To achieve the $80\%$ and $5\%$ thresholds, a single hash bit is not enough. We must build a **signature** of length $n$, divided into $b$ **bands**, where each band consists of $r$ **rows** (so $n = b \times r$).

Two documents are considered a candidate pair (they "collide") if they match *perfectly* in **at least one band**.
The probability of this happening is modeled by the S-curve equation:


$$P(\text{collision}) = 1 - (1 - p^r)^b$$

We need to solve for integer values of $b$ and $r$ that satisfy your strict constraints:

1. $$1 - (1 - 0.7048^r)^b \ge 0.80$$


2. $$1 - (1 - 0.6310^r)^b \le 0.05$$



### Step 5: Hyperparameter Tuning (Math Justification)

Because the gap between $p_1$ ($0.7048$) and $p_2$ ($0.6310$) is relatively small, we need a very steep S-curve, which requires large values for $b$ and $r$.

Let's test **$r = 31$**:

* Probability of a band matching at $S_c = 0.60$: $(0.7048)^{31} \approx 1.968 \times 10^{-5}$
* Probability of a band matching at $S_c = 0.40$: $(0.6310)^{31} \approx 6.264 \times 10^{-7}$

Now, we find $b$ to satisfy the $80\%$ threshold at $S_c = 0.60$:


$$1 - (1 - 1.968 \times 10^{-5})^b \ge 0.80$$

$$(1 - 1.968 \times 10^{-5})^b \le 0.20$$

$$b \ge \frac{\ln(0.20)}{\ln(1 - 1.968 \times 10^{-5})} \approx 81,695$$

Let's round up to **$b = 81,700$** and verify the $5\%$ threshold at $S_c = 0.40$:


$$P(\text{collision} \mid S_c = 0.40) = 1 - (1 - 6.264 \times 10^{-7})^{81700}$$


Using the approximation $(1 - x)^y \approx e^{-xy}$ for small $x$:


$$P \approx 1 - e^{-81700 \times 6.264 \times 10^{-7}} = 1 - e^{-0.0511} \approx 0.0498$$


Since $0.0498 \le 0.05$, **this satisfies your constraint perfectly.**

### Step 6: Final Signature Generation

To implement this system, you will execute the following:

1. Generate **$2,532,700$** random hyperplanes (vectors).
2. Compute the dot product of your text vector against every hyperplane.
3. If the result is positive, append a `1`; if negative, append a `0`.
4. Split this massive binary signature into **$81,700$** bands, each containing **$31$** bits.
5. Hash each band into a series of hash tables. If two text signatures fall into the exact same bucket in *any* of the $81,700$ hash tables, they are flagged as approximate nearest neighbors.

### Summary of Explicit Parameters

* **LSH Function:** Random Projection (SimHash)
* **Rows per band ($r$):** 31
* **Number of bands ($b$):** 81,700
* **Total Signature Size ($n$):** 2,532,700 bits ($\approx 316$ KB per document)

*Note on Practicality:* While the math strictly dictates these hyperparameters to fulfill your exact probability thresholds, a signature size of 2.5 million bits per document is exceptionally large for standard LSH applications. In real-world software engineering, practitioners often relax strict probability boundaries (e.g., allowing a higher false positive rate at $S_c = 0.40$) to reduce the signature size to a more computationally feasible length, such as 256 to 1024 bits.

# Problem 1b [10 pts]

Create a function `find_similar` that takes in a text and returns the most similar Gutenberg book ID in `/mnt/data/public/gutenberg/9/9` according to the criteria set in Problem 1a using Apache Spark.